Test file: For testing plot codes, not meant for endpoint run results.

### Analytical plots

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import dtale

sns.set_theme(style="whitegrid")

In [ ]:
data = pd.read_parquet("../Dataset/Clean/Dataset_with_clusters.parquet")
sample = data.sample(10)
data.shape

In [ ]:
length_column = ['char_count','sentence_count','word_count','unique_word_count','stopword_count','stopword_ratio']
diversity_column = ['lexical_diversity','hapax_ratio']
pos_column = ['noun_count', 'verb_count', 'adj_count', 'adv_count', 'pronoun_count']
ner_column = ['person_count', 'org_count', 'gpe_count', 'event_count', 'unique_entity_count']
reading_ease = ['flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog']

colors = px.colors.qualitative.Prism

In [ ]:
article = sample.iloc[5]
article

#### 1. Heuristic / Lingusitc Feature Analysis

In [ ]:
# POS Distribution histograms
def plot_grammar_composition(article, type:str):
    """Plots the percentage of grammatical composition within given article 
    
    article: pd.Series or pd.DataFrame with all generated features - single article
    """
    
    pos_data = {"POS": pos_column, "Counts": article[pos_column]}
    
    if type == "pie":
        fig = px.pie(pos_data, values='Counts', names='POS', hole=0.4, template='ggplot2', height=400, width=700,
            title="Internal Grammatical Composition [POS]", color_discrete_sequence=colors)
    elif type=='hist' or type == "bar":
        fig = px.histogram(pos_data, x='POS', y='Counts', template='ggplot2', height=400, width=700,
            title="Internal Grammatical Composition [POS]", color_discrete_sequence=colors)
    else:
        raise ValueError("Invalid plot type")
    
    return fig
    
plot_grammar_composition(article, type='bar')

In [ ]:
def plot_sentence_flow(article_text:str):
    """
    Analyzes and plots the length of each sentence in the article.
        Helps visualize the 'pacing' of the writing.
    """
    sentences = article_text.split('.') # Simple split for independent analysis
    sent_lengths = [len(s.split()) for s in sentences if len(s.split()) > 1]
    
    fig = px.area(x=range(len(sent_lengths)), y=sent_lengths,
                  title="Sentence Pacing (Word Count per Sentence)",
                  labels={'x': 'Sentence index', 'y': 'Word Count'})
    fig.update_traces(line_color='#b36efa')
    return fig

plot_sentence_flow(article['Content'])

In [ ]:
def plot_readability_gauge(score:float):
    """Standard gauge showing the article's independent difficulty level."""
    fig = go.Figure(go.Indicator(
        mode = "gauge+number",
        value = score,
        title = {'text': "Readability (Flesch Ease)"},
        gauge = {
            'axis': {'range': [0, 100]},
            'steps': [
                {'range': [0, 30], 'color': "#ff4b4b"},  # Hard
                {'range': [30, 70], 'color': "#ffa500"}, # Standard
                {'range': [70, 100], 'color': "#00cc96"} # Easy
            ],
        }
    ))
    fig.update_layout(height=350)
    return fig

plot_readability_gauge(score=article['flesch_reading_ease'])

In [ ]:
from typing import Union

def plot_information_density(datapoint: Union[pd.Series, pd.DataFrame]):
    """Visualizes the ratio of facts (Nouns/Entities) to structural fluff."""
    wc = datapoint.get('word_count', 1)
    
    metrics = {
        'Lexical Richness': (datapoint.get('unique_word_count', 0) / wc),
        'Stopword Ratio': datapoint.get('stopword_count', 0)/ wc,
        'Subject Level (Nouns)': (datapoint.get('noun_count', 0) / wc),
        'Action Level (Verbs)': (datapoint.get('verb_count', 0) / wc),
        'Fact Density (Entities)': (datapoint.get('unique_entity_count', 0) / wc)
    }
    df_plot = pd.DataFrame(list(metrics.items()), columns=['Metric', 'Value'])
    
    fig = px.bar(df_plot, x='Value', y='Metric', orientation='h',
                 title="Information Density Profile (%)",
                 color='Metric', color_discrete_sequence=px.colors.qualitative.Pastel)
    fig.update_layout(xaxis_title="Percentage of Total Words", showlegend=False)
    return fig
        

plot_information_density(article)

In [ ]:
# def get_table_view(row: Union[pd.Series, pd.DataFrame]):
#     """ Return a clean,sorted dataframe for the Excel/DataTable view. """
#     exclude = ['Content', 'Summary', 'Embedding', 'text']
#     df_view = row.drop(labels=[c for c in exclude if c in row.index]).to_frame().reset_index()
#     df_view.columns = ["Feature Attribute", "Value"]
#     return dtale.show(df_view, inline=False)

# get_table_view(article)

In [ ]:
from st_aggrid import AgGrid, GridOptionsBuilder

def get_interactive_data_view(row: Union[pd.Series, pd.DataFrame]):
    """
    Returns an interactive AG-Grid view for a single inference datapoint.
    Provides sorting, filtering, and professional Excel-like styling.
    """
    if isinstance(row, pd.Series):
        row = row.to_frame().T
        
    # 1. Clean and Prepare Data
    exclude = ['Content', 'Summary', 'Embedding', 'text']
    df_view = row.drop(columns=[c for c in exclude if c in row.columns])
    
    # Melt to vertical view for better readability of a single article
    df_melted = df_view.T.reset_index()
    df_melted.columns = ["Feature Attribute", "Value"]
    
    # 2. Configure AG Grid Options
    gb = GridOptionsBuilder.from_dataframe(df_melted)
    gb.configure_pagination(paginationAutoPageSize=True) # Auto-height
    gb.configure_side_bar() # Enables filters/columns sidebar
    gb.configure_default_column(groupable=True, value=True, enableRowGroup=True, aggFunc='sum', editable=True)
    
    grid_options = gb.build()
    
    # 3. Use in Streamlit
    # This call would happen inside your ui/pages/heuristics.py
    return AgGrid(
        df_melted,
        gridOptions=grid_options,
        data_return_mode='AS_INPUT',
        update_mode='MODEL_CHANGED',
        fit_columns_on_grid_load=True,
        theme='balham', # Professional 'Excel' light theme
        enable_enterprise_modules=False,
        height=400,
        width='100%',
    )
        
get_interactive_data_view(article)

In [ ]:
from src.Components.Visualization import *
# Using the classes directly on article rows\n

In [ ]:
from src.Components.Visualization import (
    NewsVisualizer, KMeansVisualizer, NERVisualizer,
    BERTopicVisualizer, SummarizationVisualizer
)
import plotly.io as pio
pio.renderers.default = 'notebook' 

In [ ]:
# 1. Heuristic Analysis Visualizations
viz = NewsVisualizer()
fig1 = viz.plot_sentence_flow(article)
fig1.show()
fig2 = viz.plot_grammar_composition(article, plot_type='bar')
fig2.show()
fig3 = viz.plot_information_density(article)
fig3.show()
fig4 = viz.plot_readability_gauge(article)
fig4.show()

In [ ]:
# 2. KMeans Visualizations
kviz = KMeansVisualizer(article)
kviz.plot_cluster_keywords().show()
kviz.plot_cluster_fit_gauge().show()

In [ ]:
# 3. BERTopic Visualizations
bviz = BERTopicVisualizer(article_text=article['Content'])
bviz.plot_topic_distribution().show()
bviz.plot_topic_keywords().show()

In [ ]:
# 4. Summarization Visualizations
from src.Components.Summarization import NewsSummarizer
summarizer_mod = NewsSummarizer()
summary_text = summarizer_mod.summarize(article['Content'])

sviz = SummarizationVisualizer(article['Content'], summary_text)
sviz.plot_length_comparison().show()
sviz.plot_compression_gauge().show()

In [ ]:
# 5. NER Visualizations
from src.Components.NER import InformationExtractor
ner_mod = InformationExtractor()
ner_res = ner_mod.process_articles(article['Content'])
nviz = NERVisualizer(extraction_result=ner_res)
nviz.plot_entity_distribution_spacy().show()
nviz.plot_entity_distribution_gliner().show()
nviz.plot_keywords().show()